# float32 is OPT-IN

This notebook runs in **float32** (the switch below is `True`), because it is the notebook about float32. float32 is **off by default and is not a safe replacement for float64**.

- **Set it before import.** The switch below sets PyTensor's `floatX` flag, and PyTensor reads it when it is imported. If you change `USE_FLOAT32`, **restart the kernel** and run the notebook from the top. Setting it after `import pymc` does nothing.
- **Where it helps.** On a GPU, models with a lot of arithmetic per step, such as logistic regression and the non-centered radon model, ran 3x to 8x faster on an RTX A2000 (see `float32.ipynb`). A CPU gives different, model-dependent ratios.
- **Where it fails.** The GEV model: the step size collapses and the chains barely move. 8 Schools failed the bias gate on 2 of 13 seeds. The failure can be silent.
- **Three checks on every float32 result** (README, section "float32 on the GPU: opt-in, and check the result"):
  1. `idata.attrs["frozen_step_size"]`: orders of magnitude below the float64 value is the GEV failure;
  2. R-hat on every parameter: far above 1.01 is a failure. (With only 10 draws per chain R-hat is inflated even in float64, so compare with a float64 run of the same model; a collapsed sampler is orders of magnitude worse, not tenths.)
  3. `sample_stats["num_integration_steps"]`: far from a float64 run's (GEV: hundreds of steps on the CPU, about 1 on the GPU, against about 7).

Do **not** enable `jax_enable_x64` yourself; PyTensor manages it from `floatX`.

In [1]:
USE_FLOAT32 = True      # the switch: read the cell above, then restart the kernel to change it

import os
if USE_FLOAT32:
    os.environ["PYTENSOR_FLAGS"] = "floatX=float32,blas__ldflags=-llapack -lblas -lcblas -lm"
print("float32 requested:", USE_FLOAT32)

float32 requested: True


# float32: where it helps, and what a failure looks like
Two parts. (1) **Speed**: logistic regression and the non-centered radon model run faster in float32 on a GPU. (2) **Failure**: the GEV model under float32 shows all three warning signs.

The float64 side of every comparison is run in a **child process** (a kernel cannot switch `floatX` after import). The only thing used from `benchmarks/` is its timing script `warm_stages.py`.

In [2]:
CHAINS, DRAWS, WARMUP = 1024, 10, 25      # many short chains; WARMUP is chees_kwargs["draws"]
SEED = 42

import jax
import numpy as np
import pymc as pm
import arviz as az
import pytensor
from pfcheeshmc import sample_pf_chees_hmc

print("JAX devices:", jax.devices(), "| floatX:", pytensor.config.floatX, "| x64:", jax.config.jax_enable_x64)
import json, re, subprocess, sys, time, urllib.request, zipfile, io
from pathlib import Path
import pandas as pd

assert pytensor.config.floatX == "float32", "restart the kernel with USE_FLOAT32 = True in the first cell"
REPO = Path.cwd().resolve().parent                    # this notebook lives in examples/

JAX devices: [CudaDevice(id=0)] | floatX: float32 | x64: False


.../python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part 1: where it helps
Both models are built inline, sampled in this (float32) kernel for three calls at 1,024 chains and 10 draws; call 1 compiles, so calls 2 and 3 are the timing. The float64 baseline is `benchmarks/warm_stages.py` in a child process without the flag, with the same model, chains, draws and warmup (it builds the same graphs from `benchmarks/models.py`).

In [3]:
# Logistic regression, 1000 x 25 (data drawn here: correlated predictors, a sparse true coefficient vector)
rng = np.random.default_rng(1)
Nobs, D = 1000, 25
cov = 0.5 ** np.abs(np.subtract.outer(np.arange(D), np.arange(D)))
X = rng.multivariate_normal(np.zeros(D), cov, size=Nobs)
beta_true = np.zeros(D); beta_true[:5] = [1.0, -1.0, 0.5, -0.5, 0.25]
yb = rng.binomial(1, 1 / (1 + np.exp(-X @ beta_true)))

with pm.Model() as logit_model:
    alpha = pm.Normal("alpha", 0.0, 2.5)
    beta = pm.Normal("beta", 0.0, 1.0, shape=D)
    pm.Bernoulli("y", logit_p=alpha + pm.math.dot(X, beta), observed=yb)

In [4]:
# Radon, non-centered (posteriordb radon_mn: needs network on first use)
cache = Path.home() / ".cache" / "pf-ladder" / "radon_mn.json"
if not cache.exists():
    cache.parent.mkdir(parents=True, exist_ok=True)
    raw = urllib.request.urlopen("https://raw.githubusercontent.com/stan-dev/posteriordb/master/posterior_database/"
                                 "data/data/radon_mn.json.zip", timeout=120).read()
    cache.write_bytes(zipfile.ZipFile(io.BytesIO(raw)).read("radon_mn.json"))
d = json.loads(cache.read_text())
county = np.asarray(d["county_idx"]) - 1
floor = np.asarray(d["floor_measure"], dtype=float)
log_radon = np.asarray(d["log_radon"], dtype=float)
J = int(d["J"])

with pm.Model() as radon_model:
    sigma_y = pm.HalfNormal("sigma_y", 1.0)
    sigma_alpha = pm.HalfNormal("sigma_alpha", 1.0)
    sigma_beta = pm.HalfNormal("sigma_beta", 1.0)
    mu_alpha = pm.Normal("mu_alpha", 0.0, 10.0)
    mu_beta = pm.Normal("mu_beta", 0.0, 10.0)
    alpha_raw = pm.Normal("alpha_raw", shape=(J,))
    beta_raw = pm.Normal("beta_raw", shape=(J,))
    alpha_j = pm.Deterministic("alpha", mu_alpha + sigma_alpha * alpha_raw)
    beta_j = pm.Deterministic("beta", mu_beta + sigma_beta * beta_raw)
    pm.Normal("obs", mu=alpha_j[county] + floor * beta_j[county], sigma=sigma_y, observed=log_radon)

In [5]:
def float32_calls(model, warmup, calls=3):
    times = []
    for k in range(1, calls + 1):
        t0 = time.perf_counter()
        with model:
            sample_pf_chees_hmc(draws=DRAWS, chains=CHAINS, random_seed=k, progressbar=False,
                                chees_kwargs={"draws": warmup})
        times.append(time.perf_counter() - t0)
    return times

def float64_calls(model_name, warmup, calls=3):
    """benchmarks/warm_stages.py in a child process with the float32 flag removed; returns its per-call wall times."""
    env = {k: v for k, v in os.environ.items() if k != "PYTENSOR_FLAGS"}
    out = subprocess.run([sys.executable, "benchmarks/warm_stages.py", ".", model_name, str(CHAINS), str(DRAWS),
                          str(warmup), "--calls", str(calls)], cwd=REPO, env=env, capture_output=True, text=True)
    assert "floatX: float64" in out.stdout, out.stdout + out.stderr
    return [float(t) for t in re.findall(r"call \d+: wall\s+([\d.]+)s", out.stdout)]

rows = []
for label, model, name, W in [("logistic 1000 x 25", logit_model, "logit_1000x25", 50),
                              ("radon, non-centered", radon_model, "radon_noncentered", 300)]:
    t32 = float32_calls(model, W)
    t64 = float64_calls(name, W)
    w32, w64 = np.mean(t32[1:]), np.mean(t64[1:])        # warm calls: 2 and 3
    rows.append(dict(model=label, W=W, float64_s=round(w64, 2), float32_s=round(w32, 2), ratio=round(w64 / w32, 2)))
    print(label, "float32 calls:", np.round(t32, 2), "float64 calls:", np.round(t64, 2))
pd.DataFrame(rows).set_index("model")

Initializing ChEES-HMC using pathfinder+adapt_diag...


.../python3.13/site-packages/arviz/data/base.py:272: UserWarning: More chains (1024) than draws (10). Passed array should have shape (chains, draws, *shape)
  warnings.warn(


Initializing ChEES-HMC using pathfinder+adapt_diag...
Initializing ChEES-HMC using pathfinder+adapt_diag...


.../python3.13/site-packages/arviz/data/base.py:272: UserWarning: More chains (1024) than draws (10). Passed array should have shape (chains, draws, *shape)
  warnings.warn(
.../python3.13/site-packages/arviz/data/base.py:272: UserWarning: More chains (1024) than draws (10). Passed array should have shape (chains, draws, *shape)
  warnings.warn(


logistic 1000 x 25 float32 calls: [17.92  0.18  0.17] float64 calls: [25.25  1.43  1.44]


Initializing ChEES-HMC using pathfinder+adapt_diag...


... (3 outputs left out)

Initializing ChEES-HMC using pathfinder+adapt_diag...


.../python3.13/site-packages/arviz/data/base.py:272: UserWarning: More chains (1024) than draws (10). Passed array should have shape (chains, draws, *shape)
  warnings.warn(


radon, non-centered float32 calls: [33.16  1.98  2.  ] float64 calls: [35.57  7.39  5.52]


,W,float64_s,float32_s,ratio
model,,,,
logistic 1000 x 25,50,1.44,0.18,8.12
"radon, non-centered",300,6.46,1.99,3.25


`float64_s` and `float32_s` are the mean wall time of the two warm calls (call 1 compiles and is left out), `ratio` is float64 over float32. **The speed-up is a GPU effect**: float32 arithmetic is several times faster than float64 on most GPUs, and the README measured 7.7x (logistic) and 3.1x (radon) on an RTX A2000. The committed outputs below are from an RTX A2000; a CPU gives different, model-dependent ratios. The float64 child process builds its model from `benchmarks/models.py`, the float32 side from the cells above; the graphs are the same except for the logistic data, which are drawn differently, so compare the ratio only roughly on the logistic row.

## Part 2: what a failure looks like (GEV)
GEV at W=25, 1,024 chains, 10 draws, under float32. The model is the Port Pirie model of `GEV Port Pirie example.ipynb`. For the float64 reference, the same model and call run in a child process without the flag.

In [6]:
GEV_SRC = r'''import json, sys
import numpy as np, pymc as pm, pymc_extras as pmx, arviz as az, jax
from pfcheeshmc import sample_pf_chees_hmc
data = np.array([4.03, 3.83, 3.65, 3.88, 4.01, 4.08, 4.18, 3.80, 4.36, 3.96, 3.98, 4.69, 3.85, 3.96, 3.85, 3.93,
                 3.75, 3.63, 3.57, 4.25, 3.97, 4.05, 4.24, 4.22, 3.73, 4.37, 4.06, 3.71, 3.96, 4.06, 4.55, 3.79,
                 3.89, 4.11, 3.85, 3.86, 3.86, 4.21, 4.01, 4.11, 4.24, 3.96, 4.21, 3.74, 3.85, 3.88, 3.66, 4.11,
                 3.71, 4.18, 3.90, 3.78, 3.91, 3.72, 4.00, 3.66, 3.62, 4.33, 4.55, 3.75, 4.08, 3.90, 3.88, 3.94,
                 4.33])
p = 1 / 10
with pm.Model() as model:
    μ = pm.Normal("μ", mu=3.8, sigma=0.2)
    σ = pm.HalfNormal("σ", sigma=0.3)
    ξ = pm.TruncatedNormal("ξ", mu=0, sigma=0.2, lower=-0.6, upper=0.6)
    gev = pmx.GenExtreme("gev", mu=μ, sigma=σ, xi=ξ, observed=data)
    z_p = pm.Deterministic("z_p", μ - σ / ξ * (1 - (-np.log(1 - p)) ** (-ξ)))
'''
exec(GEV_SRC)      # defines `model` (GEV) in this float32 kernel

with model:
    idata32 = sample_pf_chees_hmc(draws=DRAWS, chains=CHAINS, random_seed=SEED, progressbar=False,
                                  chees_kwargs={"draws": WARMUP})
summary32 = az.summary(idata32, var_names=["μ", "σ", "ξ"])
signs32 = dict(frozen_step_size=idata32.attrs["frozen_step_size"],
               max_rhat=float(summary32["r_hat"].max()),
               mean_num_integration_steps=float(idata32.sample_stats["num_integration_steps"].mean()))
summary32

Initializing ChEES-HMC using pathfinder+adapt_diag...


.../python3.13/site-packages/arviz/data/base.py:272: UserWarning: More chains (1024) than draws (10). Passed array should have shape (chains, draws, *shape)
  warnings.warn(


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
μ,3.880,0.029,3.827,3.935,0.001,0.000,2561.0,2594.0,56.42
σ,0.211,0.027,0.168,0.264,0.001,0.000,2561.0,2593.0,53.27
ξ,-0.038,0.101,-0.217,0.150,0.002,0.002,2561.0,2602.0,57.97


In [7]:
# the float64 reference: same model, same call, child process without the flag
CHILD = GEV_SRC + f'''
with model:
    idata = sample_pf_chees_hmc(draws={DRAWS}, chains={CHAINS}, random_seed={SEED}, progressbar=False,
                                chees_kwargs={{"draws": {WARMUP}}})
s = az.summary(idata, var_names=["μ", "σ", "ξ"])
print("RESULT", json.dumps(dict(frozen_step_size=idata.attrs["frozen_step_size"], max_rhat=float(s["r_hat"].max()),
      mean_num_integration_steps=float(idata.sample_stats["num_integration_steps"].mean()),
      floatX=str(__import__("pytensor").config.floatX))))
'''
env = {k: v for k, v in os.environ.items() if k != "PYTENSOR_FLAGS"}
env["PYTENSOR_FLAGS"] = "blas__ldflags=-llapack -lblas -lcblas -lm"
out = subprocess.run([sys.executable, "-c", CHILD], env=env, capture_output=True, text=True)
signs64 = json.loads(re.search(r"RESULT (.*)", out.stdout).group(1))
assert signs64.pop("floatX") == "float64"
pd.DataFrame({"float64 (child)": signs64, "float32 (this kernel)": signs32})

,float64 (child),float32 (this kernel)
frozen_step_size,0.520043,0.011352
max_rhat,1.890000,57.970000
mean_num_integration_steps,7.100000,1.400000


The three signs, side by side:

1. **Frozen step size**: far below the float64 value (float64 settles near 0.5 for this model; about 1e-2 or below in float32; the README measured 3e-5 to 9e-3).
2. **R-hat** far above its float64 value. Split R-hat is inflated at 10 draws per chain even in float64, so judge the gap, not 1.01 (the README measured 57 to 112 in float32).
3. **`num_integration_steps`**: far from float64's. With a tiny step size the trajectory can be hundreds of steps long (the CPU run) or collapse to about 1 step (the GPU run above); either way the chains barely move.

Be clear about what these mean. The chains barely move from their Pathfinder starting points, so a comparison of the ensemble mean against a long NUTS run (`b2_max` in `benchmarks/bias.py`) **can still pass while the sampler is not sampling**: the starts are already close to the posterior. Do not accept a float32 result on `b2_max` alone; look at the step size and R-hat.

The mitigation is not part of this notebook.